IMPORTS

In [129]:
#%pip uninstall networkx python-louvain community
#%pip install python-louvain networkx cdlib
#%pip install scikit-posthocs
#%pip install requests
import os
import re
import warnings
import textwrap
from pathlib import Path
from collections import Counter
from itertools import combinations
 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
from scipy.stats import spearmanr, chi2_contingency, kruskal, mannwhitneyu
from urllib.parse import quote
warnings.filterwarnings("ignore")

CONFIG

In [ ]:
DATA_PATH    = "retraction_watch.csv"   
OUTPUT_DIR   = "figures3"                
SCIMAGO_PATH = "scimagojr-country.csv"          
 
EXPLORE_SUBJECTS = False
 
BIOMEDICAL_SUBJECTS = [
    #"(BLS) Genetics",
    "(BLS) Neuroscience",
    "(HSC) Biostatistics/Epidemiology",
    "(HSC) Radiology/Imaging",
    "(PHY) Nanotechnology"

]

REASON_CATEGORIES = {
    "Data and Results Issues": [
        "Unreliable Results",
        "Unreliable Results and/or Conclusions",
        "Concerns/Issues About Data",
        "Error in Data",
        "Original Data not Provided",
        "Unreliable Data",
        "Error in Results and/or Conclusions",
        "Error in Analyses",
        "Error in Methods",
        "Results Not Reproducible",
        "Concerns/Issues About Results",
        "Concerns/Issues about Results and/or Conclusions",
        "Concerns/Issues about Article",
        "Concerns/Issues about Methods",
        "Error in Text",
        "Error in Materials (General)",
        "Error in Materials",
        "Error in Cell Lines/Tissues",
        "Contamination of Cell Lines/Tissues",
        "Contamination of Materials (General)",
        "Contamination of Materials",
        "Contamination of Reagents",
        "Manipulation of Results",
        "Manipulation of Data",
        "Unreliable Image",
        "Error in Image",
        "Sabotage of Materials/Methods"
    ],

    "Computer-Generated Content and AI": [
        "Computer-Aided Content or Computer-Generated Content"
        ],

    "Authorship and Ethical Concerns": [
        "Concerns/Issues About Authorship",
        "Concerns/Issues about Animal Welfare",
        "Concerns/Issues about Human Subject Welfare",
        "Concerns/Issues About Authorship/Affiliation",
        "Misconduct by Author",
        "Ethical Violations by Author",
        "Ethical Violations by Company/Institution/Third Party",
        "False/Forged Authorship",
        "False/Forged Affiliation",
        "Conflict of Interest",
        "Lack of Approval from Author",
        "Lack of Approval from Company/Institution",
        "Lack of Approval from Third Party",
        "Lack of IRB/IACUC Approval and/or Compliance",
        "Informed/Patient Consent - None/Withdrawn",
        "Author Unresponsive",
        "False Affiliation",
        "Informed/Patient Consent - None/Withdrawn"
        
    ],
    "Plagiarism and Duplication": [
        "Duplication of Image",
        "Duplication of Article",
        "Duplication of Data",
        "Duplication of Text",
        "Plagiarism of Article",
        "Euphemisms for Plagiarism",
        "Plagiarism of Text",
        "Plagiarism of Image",
        "Plagiarism of Data",
        "Euphemisms for Duplication",
        "Duplication of/in Article",
        "Duplication of/in Image",
        "Plagiarism of/in Article"
    ],
    "Image Manipulation and Fabrication": [
        "Manipulation of Images",
        "Concerns/Issues About Image",
        "Falsification/Fabrication of Image"
    ],
    "Investigations and Findings": [
        "Investigation by Journal/Publisher",
        "Investigation by Company/Institution",
        "Investigation by Third Party",
        "Investigation by ORI",
        "Misconduct - Official Investigation/Finding",
        "Investigation by Office of Research Integrity"
    ],
    "Peer Review and Editorial Issues": [
        "Fake Peer Review",
        "Rogue Editor",
        "Concerns/Issues with Peer Review",
        "Taken via Peer Review",
        "Compromised Peer Review",
        "Concerns/Issues about Peer Review"
    ],
    "Misconduct and Fraud": [
        "Paper Mill",
        "Falsification/Fabrication of Data",
        "Falsification/Fabrication of Results",
        "Misconduct by Third Party",
        "Misconduct by Company/Institution",
        "Misconduct – Official Investigation/Finding",
        "Euphemisms for Misconduct",
        "Randomly Generated Content",
        "Ethical Violations by Third Party",
        "Misconduct - Official Investigation(s) and/or Finding(s)"
    ],
    "Procedural and Legal Issues": [
        "Legal Reasons/Legal Threats",
        "Legal Reasons and/or Threats",
        "Criminal Proceedings",
        "Civil Proceedings",
        "Nonpayment of Fees/Refusal to Pay",
        "Publishing Ban"
    ],
    "Withdrawal and Retraction Notices": [
        "Withdrawal",
        "Retract and Replace",
        "Temporary Removal",
        "Date of Retraction/Other Unknown",
        "Notice - Limited or No Information",
        "Notice – Limited or No Information",
        "Notice - Lack of",
        "Notice – Lack of",
        "Notice - Unable to Access via current resources",
        "Notice – Unable to Access via current resources",
        "Upgrade/Update of Prior Notice",
        "Updated to Retraction",
        "Notice - No/Limited Information",
        "Updated to Expression of Concern",
        "Upgrade/Update of Prior Notice(s)"

    ],
    "Complaints and Objections": [
        "Objections by Third Party",
        "Objections by Author(s)",
        "Objections by Company/Institution",
        "Complaints about Author",
        "Complaints about Third Party",
        "Complaints about Company/Institution"  
    ],
    "Institutional and Policy Issues": [
        "Breach of Policy by Author",
        "Lack of IRB/IACUC Approval",
        "Concerns/Issues about Third Party Involvement",
        "Concerns/Issues about Referencing/Attributions",
        "Cites Retracted Work"
    ],
    "Miscellaneous": [
        "Copyright Claims",
        "Transfer of Copyright and/or Ownership",
        "Error by Journal/Publisher",
        "Error by Third Party",
        "Date of Article and/or Notice Unknown",
        "Duplicate Publication through Error by Journal/Publisher",
        "Duplication of Content through Error by Journal/Publisher",
        "Bias Issues or Lack of Balance",
        "Doing the Right Thing",
        "Salami Slicing",
        "Miscommunication by Author",
        "Miscommunication by Journal/Publisher",
        "Miscommunication by Company/Institution",
        "Miscommunication by Third Party",
        "Miscommunication with/by Author",
        "Miscommunication with/by Company/Institution",
        "Miscommunication with/by Journal/Publisher",
        "Miscommunication with/by Third Party",
        "Taken from Dissertation/Thesis",
        "Taken via Translation",
        "Not Presented at Conference",
        "Withdrawn to Publish in Different Journal",
        "Sabotage of Materials (Miscellaneous)",
        "Sabotage of Methods (Miscellaneous)"
        "EOC Lifted",
        "Hoax Paper",
        "No Further Action",
        "Nonpayment of Fees and/or Refusal to Pay",
        "Original Data and/or Images not Provided and/or not Available",
        "Withdrawn as Out of Date",
        "Removed",
        "Updated to Correction"

    ],
}

HINDAWI_SENSITIVITY = False
 
PALETTE = [
    "#264653", "#2A9D8F", "#E9C46A", "#F4A261",
    "#E76F51", "#457B9D", "#A8DADC", "#6D6875",
    "#B5838D", "#606C38", "#DDA15E", "#BC6C25",
    "#8D99AE"
]
 
def set_style():
    plt.rcParams.update({
        "figure.dpi"           : 150,
        "savefig.dpi"          : 300,
        "figure.facecolor"     : "white",
        "axes.facecolor"       : "#F8F8F6",
        "axes.edgecolor"       : "#BBBBBB",
        "axes.linewidth"       : 0.8,
        "axes.spines.top"      : False,
        "axes.spines.right"    : False,
        "axes.grid"            : False,
        "axes.grid.axis"       : "y",
        "grid.color"           : "#E2E2E2",
        "grid.linewidth"       : 0.5,
        "font.family"          : "sans-serif",
        "font.size"            : 11,
        "axes.titlesize"       : 13,
        "axes.titleweight"     : "bold",
        "axes.titlepad"        : 10,
        "axes.labelsize"       : 11,
        "xtick.labelsize"      : 10,
        "ytick.labelsize"      : 10,
        "legend.fontsize"      : 9,
        "legend.framealpha"    : 0.9,
        "legend.edgecolor"     : "#CCCCCC",
        "legend.frameon"       : True,
    })
 
def savefig(name, fig=None):
    """Save as PDF (vector) + PNG (raster) at 300 dpi."""
    Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
    target = fig if fig else plt
    target.tight_layout()
    for ext in ("pdf", "png"):
        target.savefig(f"{OUTPUT_DIR}/{name}.{ext}", bbox_inches="tight")
    plt.close("all")
    print(f"    → {name}.pdf / .png")

READ AND FILTER

In [ ]:
def phase1_load(path: str) -> pd.DataFrame | None:
    print("\n══════════════════════════════════════════")
    print("  Phase 1 — Data ingestion & subject filter")
    print("══════════════════════════════════════════")
 
    df = pd.read_csv(path, low_memory=False)
    print(f"  Total records in CSV          : {len(df):>8,}")
 
    df = df[df["RetractionNature"].str.strip() == "Retraction"].copy()
    print(f"  After RetractionNature filter : {len(df):>8,}")
 
    if EXPLORE_SUBJECTS:
        subjects = (
            df["Subject"].dropna()
            .str.split(";").explode()
            .str.strip().unique()
        )
        print("\n  ── Unique Subject values in the retraction-only subset ──\n")
        for s in sorted(subjects):
            print(f"    {s}")
    
        return None
 
    if not BIOMEDICAL_SUBJECTS:
        raise ValueError("BIOMEDICAL_SUBJECTS is empty. Add tags and set EXPLORE_SUBJECTS=False.")
 
    pattern = "|".join(re.escape(s) for s in BIOMEDICAL_SUBJECTS)
    mask    = df["Subject"].fillna("").str.contains(pattern, case=False, regex=True)
    df      = df[mask].copy()
    print(f"  After biomedical subject filter: {len(df):>7,}")
    df["Subject"] = df["Subject"].str.replace("(BLS) Neuroscience ", "Neuroscience").str.replace("(HSC) Biostatistics/Epidemiology ", "Biostatistics").str.replace("(HSC) Radiology/Imaging ", "Imaging/Radiology").str.replace("(PHY) Nanotechnology ", "Nanotechnology")
    print(f"  After biomedical subject filter: {len(df):>7,}")
    return df



CLEAN AND FEATURE ENG

In [132]:
def _map_reasons(raw_str: str | float) -> list[str]:
    """
    Convert a semicolon-separated Retraction Watch Reason string into
    a list of user-defined category labels.
    Makes a list of the categories that don't match any of the predefined categories → "Other / unknown"
    """
    if pd.isna(raw_str):
        return ["Other / unknown"]
    tokens = [r.strip() for r in str(raw_str).split(";") if r.strip()]
    cats   = set()
    for tok in tokens:
        matched = False
        for cat, members in REASON_CATEGORIES.items():
            if any(tok.lower() == m.lower() for m in members):
                cats.add(cat)
                matched = True
                break
        if not matched:
            print(f"  Unmatched reason: {tok}")
            cats.add("Other / unknown")
    return list(cats) if cats else ["Other / unknown"]
 
 
def phase2_clean(df: pd.DataFrame) -> pd.DataFrame:
    print("\n══════════════════════════════════════════")
    print("  Phase 2 — Cleaning & feature engineering")
    print("══════════════════════════════════════════")
 
    df["OriginalPaperDate"] = pd.to_datetime(df["OriginalPaperDate"], errors="coerce")
    df["RetractionDate"]    = pd.to_datetime(df["RetractionDate"],    errors="coerce")
    df["pub_year"]          = df["OriginalPaperDate"].dt.year.astype("Int64")
    df["retraction_year"]   = df["RetractionDate"].dt.year.astype("Int64")
    df["time_to_retraction_days"]  = (df["RetractionDate"] - df["OriginalPaperDate"]).dt.days
    df["time_to_retraction_years"] = df["time_to_retraction_days"] / 365.25
 
    for col in ("RetractionDOI", "OriginalPaperDOI"):
        df[col] = df[col].fillna("").str.strip()
        df[col] = df[col].where(~df[col].isin({"unavailable","Unavailable",""}), other=np.nan)
    for col in ("RetractionPubMedID", "OriginalPaperPubMedID"):
        df[col] = pd.to_numeric(df[col], errors="coerce").replace(0, np.nan)
 
    df["has_retraction_doi"] = df["RetractionDOI"].notna()
    df["paywalled"] = (
        df["Paywalled"].fillna("").str.strip().str.upper().map({"YES":True,"NO":False})
    )
    df["ReasonCategories"] = df["Reason"].apply(_map_reasons)
    df["author_count"] = (
        df["Author"].fillna("").str.split(";")
        .apply(lambda x: len([a for a in x if a.strip()]))
        .replace(0, np.nan)
    )
    df["is_hindawi"] = df["Publisher"].fillna("").str.contains("Hindawi", case=False)
    df["pub_decade"] = ((df["pub_year"] // 10) * 10).astype("Int64")
 
    before = len(df)
    df = df[~(df["time_to_retraction_days"] < 0)].copy()
    print(f"  Dropped {before-len(df)} rows with negative TTR")
    print(f"  Clean records: {len(df):,}  |  Hindawi: {df['is_hindawi'].sum():,}")
    return df
 
 
def build_reasons_long(df):
    """One row per (Record ID × reason category), including all feature columns."""
    return (
        df[["Record ID","ReasonCategories","retraction_year","pub_year","pub_decade",
            "paywalled","time_to_retraction_years","author_count","is_hindawi"]]
        .explode("ReasonCategories")
        .rename(columns={"ReasonCategories":"Category"})
    )

TEMPORAL TRENDS

In [ ]:
def phase3a_temporal(df):
    print("\n══ Phase 3a — Temporal trends ══")
    set_style()
 
    annual = (
        df.groupby("retraction_year").size().reset_index(name="count")
        .query("1980 <= retraction_year <= 2026")
    )
    if HINDAWI_SENSITIVITY:
        ann_nh = (
            df[~df["is_hindawi"]].groupby("retraction_year").size()
            .reset_index(name="count_no_hindawi")
            .query("1980 <= retraction_year <= 2026")
        )
        annual = annual.merge(ann_nh, on="retraction_year", how="left")
 
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.fill_between(annual["retraction_year"], annual["count"], alpha=0.12, color=PALETTE[0])
    ax.plot(annual["retraction_year"], annual["count"],
            color=PALETTE[0], lw=2.5, marker="o", ms=4, label="All retractions")
    if HINDAWI_SENSITIVITY and "count_no_hindawi" in annual.columns:
        ax.plot(annual["retraction_year"], annual["count_no_hindawi"],
                color=PALETTE[3], lw=2, ls="--", marker="s", ms=3, label="Excl. Hindawi")
    ax.axvline(2023, color=PALETTE[4], lw=1.2, ls="--", alpha=0.8, label="Hindawi mass retraction")
    ax.set_xlabel("Year of retraction"); ax.set_ylabel("Number of retractions")
    ax.set_title("Annual biomedical retractions (1980–2026)")
    ax.legend(fontsize=9)
    savefig("fig01_annual_retractions")
 
    ttr   = df["time_to_retraction_years"].dropna()
    ttr   = ttr[(ttr >= 0) & (ttr <= 30)]
    dec_v = (
        df.assign(dl=df["pub_decade"].astype(str)+"s")
        [["dl","time_to_retraction_years"]].dropna()
        .query("0 <= time_to_retraction_years <= 30")
    )
    avail = sorted(dec_v["dl"].unique())
 
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].hist(ttr, bins=45, color=PALETTE[1], edgecolor="white", lw=0.4, alpha=0.9)
    axes[0].axvline(ttr.median(), color=PALETTE[4], lw=2, ls="--", label=f"Median: {ttr.median():.1f} yr")
    axes[0].axvline(ttr.mean(),   color=PALETTE[0], lw=2, ls=":",  label=f"Mean: {ttr.mean():.1f} yr")
    axes[0].set_xlabel("Time to retraction (years)"); axes[0].set_ylabel("Count")
    axes[0].set_title("Time-to-retraction distribution"); axes[0].legend()
    sns.violinplot(data=dec_v, x="dl", y="time_to_retraction_years",
                   order=avail, palette=PALETTE[:len(avail)],
                   inner="quartile", linewidth=0.8, ax=axes[1])
    axes[1].set_xlabel("Publication decade"); axes[1].set_ylabel("Time to retraction (years)")
    axes[1].set_title("TTR by publication decade")
    savefig("fig02_time_to_retraction")
 
    heat = (
        df.groupby(["pub_year","retraction_year"]).size().unstack(fill_value=0)
    )
    heat = heat.loc[
        heat.index.isin(range(1980,2026)),
        [c for c in range(1990,2027) if c in heat.columns]
    ]
    fig, ax = plt.subplots(figsize=(13, 7))
    sns.heatmap(heat, cmap="Blues", linewidths=0, ax=ax,
                cbar_kws={"label":"No. of retractions","shrink":0.55})
    ax.set_xlabel("Year of retraction"); ax.set_ylabel("Year of original publication")
    ax.set_title("Publication year vs. retraction year — density heatmap")
    #savefig("fig03_pub_vs_retraction_heatmap")
 

REASONS ANALYSIS

In [134]:
def phase3b_reasons(df, reasons_long):
    print("\n══ Phase 3b — Reasons analysis ══")
    set_style()
 
    freq      = reasons_long["Category"].value_counts()
    cat_order = freq.index.tolist()
    cat_col   = {c: PALETTE[i % len(PALETTE)] for i, c in enumerate(cat_order)}
 
    fig, ax = plt.subplots(figsize=(9, 7))
    cols = [cat_col[c] for c in cat_order[::-1]]
    bars = ax.barh(cat_order[::-1], freq.values[::-1], color=cols, alpha=0.88)
    ax.bar_label(bars, labels=[f"{v:,}" for v in freq.values[::-1]], padding=5, fontsize=9.5)
    ax.set_xlim(0, freq.max()*1.18); ax.set_xlabel("Number of retractions")
    ax.set_title("Frequency per retraction reason")
    ax.grid(axis="x", visible=False); ax.grid(axis="y", visible=False)
    savefig("fig04_reason_frequency")
 
    cooc = pd.DataFrame(0, index=cat_order, columns=cat_order)
    for cats in df["ReasonCategories"]:
        for a in cats:
            for b in cats:
                if a in cooc.index and b in cooc.columns:
                    cooc.loc[a, b] += 1
    np.fill_diagonal(cooc.values, 0)
    mask = np.triu(np.ones_like(cooc, dtype=bool), k=1)
    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(cooc, mask=mask, cmap="YlOrBr", annot=True, fmt="d",
                linewidths=0.4, ax=ax, cbar_kws={"label":"Co-occurrences","shrink":0.55})
    ax.set_title("Reason co-occurrence matrix")
    plt.xticks(rotation=40, ha="right")
    savefig("fig05_reason_cooccurrence")
 
    top6 = cat_order[:6]
    rt = (
        reasons_long[reasons_long["Category"].isin(top6)]
        .groupby(["retraction_year","Category"]).size().reset_index(name="count")
        .query("1980 <= retraction_year <= 2026")
    )
    fig, ax = plt.subplots(figsize=(10, 5))
    for i, cat in enumerate(top6):
        sub = rt[rt["Category"]==cat]
        ax.plot(sub["retraction_year"], sub["count"],
                label=textwrap.shorten(cat,32), color=PALETTE[i], lw=2.2, marker="o", ms=3)
    ax.set_xlabel("Year"); ax.set_ylabel("Number of retractions")
    ax.set_title("Top-6 reason categories over time")
    ax.legend(fontsize=8.5, ncol=2)
    savefig("fig06_reason_trends")
 

GEOGRAPHIC ANALYSIS

In [ ]:
def _make_iso_lookup():
    try:
        import pycountry
        from thefuzz import process as fp
    except ImportError:
        raise ImportError("pip install pycountry thefuzz")
    _n2i = {c.name: c.alpha_3 for c in pycountry.countries}
    _n2i.update({c.alpha_2: c.alpha_3 for c in pycountry.countries})
    _n2i.update({c.alpha_3: c.alpha_3 for c in pycountry.countries})
    _aliases = {
        "United States":"USA","USA":"USA","U.S.A.":"USA",
        "UK":"GBR","United Kingdom":"GBR",
        "China":"CHN","People's Republic of China":"CHN",
        "Iran":"IRN","South Korea":"KOR","Republic of Korea":"KOR",
        "Taiwan":"TWN","Russia":"RUS","Czech Republic":"CZE",
        "Venezuela":"VEN","Bolivia":"BOL",
    }
    _cache = {}
    def to_iso3(name):
        if name in _cache: return _cache[name]
        if name in _aliases: _cache[name]=_aliases[name]; return _aliases[name]
        try:
            r = pycountry.countries.lookup(name).alpha_3
        except LookupError:
            match, score = fp.extractOne(name, list(_n2i.keys()))
            r = _n2i[match] if score >= 90 else None
        _cache[name]=r; return r
    def to_name(code):
        try: return pycountry.countries.get(alpha_3=code).name
        except: return code
    return to_iso3, to_name

def phase3c_geographic(df):
    """
    Fixes:
    - Choropleth uses sqrt color scale so China doesn't dominate.
    - Top-20 normalized bar filters to countries with ≥100,000 total publications
      (major research nations only) to avoid Ethiopia/Iraq artifacts.
    - Raw count map added alongside normalized map.
    """
    print("\n══ Phase 3c — Geographic analysis ══")
    try:
        to_iso3, to_name = _make_iso_lookup()
    except ImportError as e:
        print(f"  {e}"); return None, None
    set_style()
 
    geo = (
        df[["Record ID","Country","retraction_year","ReasonCategories","is_hindawi"]]
        .assign(Country=df["Country"].str.split(";")).explode("Country")
    )
    geo["Country"] = geo["Country"].str.strip()
    invalid = {"unknown","not available","na","n/a","none",""}
    geo = geo[geo["Country"].str.lower().notna() & (~geo["Country"].str.lower().isin(invalid))].copy()
    geo["iso3"] = geo["Country"].apply(to_iso3)
    geo = geo[geo["iso3"].notna()].copy()
 
    country_counts = geo.groupby("iso3").size().reset_index(name="retraction_count")
    country_counts["country_name"] = country_counts["iso3"].apply(to_name)

    try:
        import plotly.express as px
        import plotly.graph_objects as go
 
        cc = country_counts.copy()
        cc["sqrt_count"] = np.sqrt(cc["retraction_count"])
        cc["hover"] = cc.apply(
            lambda r: f"<b>{r['country_name']}</b><br>"
                      f"Retractions: {int(r['retraction_count']):,}<br>"
                      f"√count (color scale): {r['sqrt_count']:.1f}",
            axis=1,
        )
        fig7 = px.choropleth(
            cc, locations="iso3", color="sqrt_count",
            color_continuous_scale="Blues",
            hover_name="country_name",
            custom_data=["retraction_count"],
            labels={"sqrt_count": "√(retractions)"},
            title="Global biomedical retractions (colour = √count, hover = raw count)",
        )
        fig7.update_traces(
            hovertemplate="<b>%{hovertext}</b><br>Retractions: %{customdata[0]:,}<extra></extra>"
        )
        fig7.add_annotation(
            text="Colour scale is √(count) to prevent China from washing out all other countries.",
            xref="paper", yref="paper", x=0.5, y=-0.05,
            showarrow=False, font=dict(size=10, color="gray"),
            xanchor="center",
        )
        fig7.update_layout(
            geo=dict(showframe=False, showcoastlines=True, coastlinecolor="#AAAAAA"),
            font_family="Arial", title_font_size=14,
            margin=dict(l=0, r=0, t=50, b=40),
            coloraxis_colorbar=dict(
                title="√(retractions)",
                tickvals=[np.sqrt(x) for x in [1, 10, 100, 500, 1000, 3000]],
                ticktext=["1", "10", "100", "500", "1,000", "3,000"],
            ),
        )
        Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
        fig7.write_html(f"{OUTPUT_DIR}/fig07_choropleth.html")
        print("    → fig07_choropleth.html  (sqrt color scale, real counts in hover)")
    except ImportError:
        print("  plotly not installed.")
 
    # ── Fig 08a: Top-20 by raw count ──────────────────────────────────────────
    top20_raw = country_counts.nlargest(20, "retraction_count").copy()
    fig, ax = plt.subplots(figsize=(9, 8))
    bars = ax.barh(top20_raw["country_name"][::-1].values,
                   top20_raw["retraction_count"][::-1].values,
                   color=PALETTE[0], alpha=0.85)
    ax.bar_label(bars, labels=[f"{v:,}" for v in top20_raw["retraction_count"][::-1].values],
                 padding=4, fontsize=9)
    ax.set_xlim(0, top20_raw["retraction_count"].max() * 1.18)
    ax.set_xlabel("Number of retractions")
    ax.set_title("Top 20 countries")
    ax.grid(axis="x", visible=False); ax.grid(axis="y", visible=False)
    savefig("fig08a_top20_raw_counts")
 
    # ── Fig 08b: Normalized rate — major research nations only ────────────────
    MAJOR_NATION_MIN_DOCS = 100_000
 
    if Path(SCIMAGO_PATH).exists():
        try:
            sc = pd.read_csv(SCIMAGO_PATH)
            sc["iso3"] = sc["Country"].apply(to_iso3)
            sc = sc.dropna(subset=["iso3"])
            sc_agg = sc.groupby("iso3")["Documents"].sum().reset_index()
 
            sc_major = sc_agg[sc_agg["Documents"] >= MAJOR_NATION_MIN_DOCS].copy()
            print(f"  Major research nations (≥{MAJOR_NATION_MIN_DOCS:,} docs): {len(sc_major)}")
 
            merged = country_counts.merge(sc_major, on="iso3", how="inner")
            merged["rate_per_10k"] = merged["retraction_count"] / merged["Documents"] * 10_000
 
            def _ci(row, n=2000):
                p = row["retraction_count"] / row["Documents"]
                s = np.random.binomial(int(row["Documents"]), p, n) / row["Documents"] * 10_000
                return np.percentile(s, [2.5, 97.5])
            cis = merged.apply(_ci, axis=1)
            merged["ci_low"]  = [c[0] for c in cis]
            merged["ci_high"] = [c[1] for c in cis]
            merged["country_name"] = merged["iso3"].apply(to_name)
 
            top20_norm = merged.nlargest(20, "rate_per_10k").copy()
 
            fig, ax = plt.subplots(figsize=(9, 8))
            bars = ax.barh(top20_norm["country_name"][::-1].values,
                           top20_norm["rate_per_10k"][::-1].values,
                           color=PALETTE[1], alpha=0.85)
            ax.errorbar(
                top20_norm["rate_per_10k"][::-1].values,
                top20_norm["country_name"][::-1].values,
                xerr=[(top20_norm["rate_per_10k"]-top20_norm["ci_low"])[::-1].values,
                      (top20_norm["ci_high"]-top20_norm["rate_per_10k"])[::-1].values],
                fmt="none", color="#333", lw=1.2, capsize=3,
            )
            '''
            for i, (_, row) in enumerate(top20_norm[::-1].reset_index(drop=True).iterrows()):
                ax.text(top20_norm["rate_per_10k"][::-1].iloc[i] * 1.01, i,
                        f"n={int(row['retraction_count']):,}", va="center", fontsize=8.5, color="#444")
            '''
            ax.set_xlim(0, top20_norm["rate_per_10k"].max() * 1.35)
            ax.set_xlabel("Retractions per 10,000 publications")
            ax.set_title(
                f"Top 20 countries (normalized retraction rate)\n"
                f"(Only: ≥{MAJOR_NATION_MIN_DOCS:,} total publications)"
            )
            ax.grid(axis="x", visible=False); ax.grid(axis="y", visible=False)
            savefig("fig08b_top20_normalized_major_nations")
            country_counts = merged   # use normalized for downstream
 
        except Exception as e:
            print(f"  SCImago merge failed ({e})")
    else:
        print(f"  {SCIMAGO_PATH} not found — skipping normalized chart.")
 
    # ── Figs 09, 10 unchanged ─────────────────────────────────────────────────
    top10_iso = country_counts.nlargest(10, "retraction_count")["iso3"].tolist()
    cr = (
        geo[geo["iso3"].isin(top10_iso)]
        .explode("ReasonCategories").rename(columns={"ReasonCategories":"Category"})
    )
    ct = cr.groupby(["iso3","Category"]).size().unstack(fill_value=0)
    ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
    ct_pct.index = ct_pct.index.map(to_name)
    fig, ax = plt.subplots(figsize=(14, 6))
    sns.heatmap(ct_pct, cmap="YlOrRd", annot=True, fmt=".0f",
                linewidths=0.4, ax=ax,
                cbar_kws={"label":"% of country's retractions","shrink":0.55})
    ax.set_title("Retraction reason heatmap (Top 10 countries by total retractions)")
    plt.xticks(rotation=38, ha="right", fontsize=9)
    savefig("fig09_country_reason_heatmap")
 
    top5 = country_counts.nlargest(5, "retraction_count")["iso3"].tolist()
    geo_yr = (
        geo[geo["iso3"].isin(top5)]
        .groupby(["iso3","retraction_year"]).size().reset_index(name="count")
        .query("1980 <= retraction_year <= 2026")
    )
    fig, ax = plt.subplots(figsize=(10, 5))
    for i, iso in enumerate(top5):
        sub = geo_yr[geo_yr["iso3"]==iso]
        ax.plot(sub["retraction_year"], sub["count"],
                label=to_name(iso), color=PALETTE[i], lw=2.2, marker="o", ms=3)
    ax.set_xlabel("Year"); ax.set_ylabel("Number of retractions")
    ax.set_title("Annual retractions (Top 5 countries)"); ax.legend()
    savefig("fig10_country_trends")
 
    return country_counts, geo

JOURNALS, PUBLISHERS, AUTHORS

In [ ]:
def phase4_journals_authors(df):
    print("\n══ Phase 4 — Journals, publishers & authors ══")
    set_style()
 
    top_j = df["Journal"].value_counts().head(20)
    fig, ax = plt.subplots(figsize=(9, 8))
    bars = ax.barh(top_j.index[::-1], top_j.values[::-1], color=PALETTE[2], alpha=0.85)
    ax.bar_label(bars, labels=[f"{v:,}" for v in top_j.values[::-1]], padding=4, fontsize=9)
    ax.set_xlim(0, top_j.max()*1.18); ax.set_xlabel("Number of retractions")
    ax.set_title("Top 20 journals by retraction count")
    ax.grid(axis="x", visible=False); ax.grid(axis="y", visible=False)
    savefig("fig11_top_journals")
 
    top_p = df["Publisher"].value_counts().head(15)
    fig, ax = plt.subplots(figsize=(9, 7))
    bars = ax.barh(top_p.index[::-1], top_p.values[::-1], color=PALETTE[3], alpha=0.85)
    ax.bar_label(bars, labels=[f"{v:,}" for v in top_p.values[::-1]], padding=4, fontsize=9)
    ax.set_xlim(0, top_p.max()*1.18); ax.set_xlabel("Number of retractions")
    ax.set_title("Top 15 publishers"); ax.grid(axis="x", visible=False); ax.grid(axis="y", visible=False)
    savefig("fig12_top_publishers")

    pay  = df["paywalled"].value_counts()
    vals = [pay.get(False,0), pay.get(True,0)]
    fig, ax = plt.subplots(figsize=(6, 5))
    wedges,_,autotexts = ax.pie(
        vals, labels=[f"Open\n(n={vals[0]:,})", f"Paywalled\n(n={vals[1]:,})"],
        autopct="%1.1f%%", colors=[PALETTE[1], PALETTE[4]],
        startangle=90, wedgeprops={"edgecolor":"white","linewidth":2},
        textprops={"fontsize":11},
    )
    for t in autotexts: t.set_fontsize(12); t.set_fontweight("bold")
    ax.set_title("Retraction notice access")
    #savefig("fig13_paywall_pie")
 
    top8 = df["Publisher"].value_counts().head(8).index
    pay_pub = (
        df[df["Publisher"].isin(top8)].groupby("Publisher")["paywalled"]
        .mean().dropna().sort_values()
    )
    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.barh([textwrap.shorten(p,35) for p in pay_pub.index],
                   pay_pub.values*100, color=PALETTE[4], alpha=0.85)
    ax.axvline(50, color="#888", lw=1, ls="--")
    ax.set_xlabel("% paywalled"); ax.set_title("Paywall rate — top 8 publishers")
    ax.set_xlim(0, 110)
    ax.bar_label(bars, labels=[f"{v*100:.0f}%" for v in pay_pub.values], padding=4, fontsize=9)
    ax.grid(axis="x", visible=False); ax.grid(axis="y", visible=False)
    #savefig("fig14_paywall_by_publisher")
 
    author_counts = df["Author"].dropna().str.split(";").explode().str.strip().value_counts()
    hist_v = author_counts.value_counts().sort_index().loc[lambda s: s.index<=15]
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(hist_v.index, hist_v.values, color=PALETTE[0], alpha=0.85, edgecolor="white", lw=0.5)
    ax.set_yscale("log"); ax.yaxis.set_major_formatter(mticker.ScalarFormatter())
    ax.set_xlabel("Retractions per author"); ax.set_ylabel("Number of authors (log scale)")
    ax.set_title("Distribution of per-author retraction counts")
    #savefig("fig15_serial_retractors")
 
    ac_df = df[["author_count","time_to_retraction_years"]].dropna()
    ac_df = ac_df[ac_df["author_count"]<=20].copy()
    bins   = [0,1,2,3,4,5,7,10,15,20]
    labels = ["1","2","3","4","5","6–7","8–10","11–15","16–20"]
    ac_df["group"] = pd.cut(ac_df["author_count"], bins=bins, labels=labels, right=True)
    gs = (
        ac_df.groupby("group", observed=True)
        .agg(count=("time_to_retraction_years","size"),
             median_ttr=("time_to_retraction_years","median"))
        .reset_index()
    )
    fig, ax1 = plt.subplots(figsize=(9, 5))
    ax2 = ax1.twinx()
    ax1.bar(gs["group"].astype(str), gs["count"], color=PALETTE[0], alpha=0.72, label="No. of retractions")
    ax2.plot(gs["group"].astype(str), gs["median_ttr"],
             color=PALETTE[4], lw=2.5, marker="D", ms=6, label="Median TTR (yr)")
    ax1.set_xlabel("Number of authors")
    ax1.set_ylabel("Number of retractions", color=PALETTE[0])
    ax2.set_ylabel("Median time to retraction (years)", color=PALETTE[4])
    ax1.set_title("Retractions and TTR by number of authors")
    h1,l1 = ax1.get_legend_handles_labels()
    h2,l2 = ax2.get_legend_handles_labels()
    ax1.legend(h1+h2, l1+l2, fontsize=9)
    ax1.grid(axis="y"); ax1.spines["top"].set_visible(False); ax2.spines["top"].set_visible(False)
    savefig("fig16_retractions_by_author_count")
 
    return author_counts

STATS TEST AND MODELS

In [ ]:
def phase5_statistics(df, reasons_long):
    print("\n══ Phase 5 — Statistical testing & modelling ══")
    set_style()
 
    #Kaplan-Meier
    try:
        from lifelines import KaplanMeierFitter
        from lifelines.statistics import logrank_test
        top4 = reasons_long["Category"].value_counts().index[:8].tolist()
        km_df = (
            reasons_long[reasons_long["Category"].isin(top4)]
            [["Record ID","Category","time_to_retraction_years"]]
            .drop_duplicates("Record ID")
            .dropna(subset=["time_to_retraction_years"])
            .query("0 <= time_to_retraction_years <= 30")
        )
        fig, ax = plt.subplots(figsize=(9, 6))
        kmf = KaplanMeierFitter()
        for i, cat in enumerate(top4):
            sub = km_df[km_df["Category"]==cat]["time_to_retraction_years"]
            if len(sub)<10: continue
            kmf.fit(sub, label=textwrap.shorten(cat,35))
            kmf.plot_survival_function(ax=ax, ci_show=True, color=PALETTE[i], lw=2.2)
        ax.set_xlabel("Time to retraction (years)"); ax.set_ylabel("Proportion not yet retracted")
        ax.set_title("Kaplan–Meier curves by reason category"); ax.legend(fontsize=9)
        savefig("fig17_km_survival")
        g1 = km_df[km_df["Category"]==top4[0]]["time_to_retraction_years"]
        g2 = km_df[km_df["Category"]==top4[1]]["time_to_retraction_years"]
        lr = logrank_test(g1, g2)
        print(f"  Log-rank {top4[0]!r} vs {top4[1]!r}: p={lr.p_value:.4f}")
    except ImportError:
        print("  lifelines not installed (pip install lifelines)")
 
    ttr_yr = (
        df.query("1980 <= retraction_year <= 2026")
        .groupby("retraction_year")["time_to_retraction_years"]
        .median().reset_index(name="median_ttr")
    )
    ttr_yr["rolling"] = ttr_yr["median_ttr"].rolling(3, center=True).mean()
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.bar(ttr_yr["retraction_year"], ttr_yr["median_ttr"], color=PALETTE[6], alpha=0.55, label="Annual median")
    ax.plot(ttr_yr["retraction_year"], ttr_yr["rolling"], color=PALETTE[0], lw=2.5, label="3-yr rolling mean")
    ax.set_xlabel("Year"); ax.set_ylabel("Median TTR (years)")
    ax.set_title("Is retraction happening faster over time?"); ax.legend()
    savefig("fig18_ttr_trend")
 
    ttr_v = df["time_to_retraction_years"].dropna().values
    ttr_v = ttr_v[(ttr_v>=0)&(ttr_v<=30)]
    boot  = [np.median(np.random.choice(ttr_v, len(ttr_v), replace=True)) for _ in range(5000)]
    ci_lo, ci_hi = np.percentile(boot, [2.5, 97.5])
    print(f"  Median TTR: {np.median(ttr_v):.2f} yr  (95% CI {ci_lo:.2f}–{ci_hi:.2f})")
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(boot, bins=60, color=PALETTE[1], edgecolor="white", lw=0.3, alpha=0.9)
    ax.axvline(np.median(ttr_v), color=PALETTE[4], lw=2,
               label=f"Observed: {np.median(ttr_v):.2f} yr")
    ax.axvspan(ci_lo, ci_hi, alpha=0.15, color=PALETTE[4],
               label=f"95% CI [{ci_lo:.2f}, {ci_hi:.2f}]")
    ax.set_xlabel("Bootstrap median TTR"); ax.set_ylabel("Frequency")
    ax.set_title("Bootstrap distribution of median TTR"); ax.legend(fontsize=9)
    savefig("fig19_bootstrap_median_ttr")

TEMPORAL LAG BY COUNTRY

In [ ]:
def phase5b_temporal_lag_by_country(df):
    """
    Improved statistical analysis:
    1. Kruskal-Wallis (already working)
    2. Dunn's post-hoc pairwise test (which country pairs differ)
    3. OLS with country dummies (interpretable coefficients)
    4. Reason-stratified TTR boxplot per country (explains the LME result)
    5. Visualise: median TTR per country + per-country reason composition side-by-side
    """
    print("\n══ Phase 5b — Temporal lag by country (improved) ══")
    try:
        to_iso3, to_name = _make_iso_lookup()
    except ImportError as e:
        print(f"  {e}"); return
    set_style()
 
    geo_ttr = (
        df[["Record ID","Country","time_to_retraction_years","ReasonCategories","retraction_year"]]
        .dropna(subset=["Country","time_to_retraction_years"])
        .query("0 <= time_to_retraction_years <= 30").copy()
    )
    geo_ttr["primary_country"] = (
        geo_ttr["Country"].str.split(";").apply(lambda x: x[0].strip() if x else None)
    )
    geo_ttr["iso3"] = geo_ttr["primary_country"].apply(to_iso3)
    geo_ttr = geo_ttr.dropna(subset=["iso3"])
    geo_ttr["primary_reason"] = geo_ttr["ReasonCategories"].apply(lambda x: x[0])
    geo_ttr["country_name"]   = geo_ttr["iso3"].apply(to_name)
 
    counts = geo_ttr["iso3"].value_counts()
    geo_ttr = geo_ttr[geo_ttr["iso3"].isin(counts[counts >= 50].index)].copy()
    print(f"  Countries with ≥50 papers: {geo_ttr['iso3'].nunique()}")
 
    # ── 1. Kruskal-Wallis ─────────────────────────────────────────────────────
    groups = [g["time_to_retraction_years"].values for _, g in geo_ttr.groupby("iso3")]
    H, p_kw = kruskal(*groups)
    print(f"  Kruskal-Wallis: H={H:.1f}, p={p_kw:.3e}")
 
    # ── 2. Dunn's post-hoc (top 10 countries for readability) ─────────────────
    try:
        import scikit_posthocs as sp
        top10_iso = geo_ttr["iso3"].value_counts().index[:10].tolist()
        dunn_df   = geo_ttr[geo_ttr["iso3"].isin(top10_iso)].copy()
        dunn_df["country_name"] = dunn_df["iso3"].apply(to_name)
        dunn = sp.posthoc_dunn(
            dunn_df, val_col="time_to_retraction_years",
            group_col="country_name", p_adjust="bonferroni",
        )
        # Plot Dunn p-value heatmap
        fig, ax = plt.subplots(figsize=(10, 9))
        sig = dunn < 0.05
        sns.heatmap(
            -np.log10(dunn + 1e-300), annot=dunn.round(3).astype(str),
            fmt="", cmap="RdYlGn", linewidths=0.5, ax=ax,
            cbar_kws={"label":"-log₁₀(p Bonferroni)", "shrink":0.6},
        )
        ax.set_title("Dunn's post-hoc pairwise test — time to retraction\n(green = p<0.05)")
        plt.xticks(rotation=35, ha="right", fontsize=8)
        plt.yticks(fontsize=8)
        savefig("fig20b_dunn_posthoc")
    except ImportError:
        print("  scikit-posthocs not installed (pip install scikit-posthocs) — skipping Dunn's")
 
    # ── 3. OLS with country dummies ───────────────────────────────────────────
    try:
        import statsmodels.formula.api as smf
        ols_df = geo_ttr.copy()
        ols_df["log_ttr"]     = np.log1p(ols_df["time_to_retraction_years"])
        ols_df["reason_code"] = ols_df["primary_reason"].astype("category").cat.codes
        ols_df["year_c"]      = ols_df["retraction_year"] - ols_df["retraction_year"].mean()
        ref = ols_df["country_name"].value_counts().index[0]
        ols_df["country_cat"] = pd.Categorical(ols_df["country_name"], categories=[ref] +
                                [c for c in ols_df["country_name"].unique() if c != ref])
        md  = smf.ols("log_ttr ~ C(country_cat) + reason_code + year_c", data=ols_df)
        res = md.fit(cov_type="HC3")   # heteroskedasticity-robust SEs
        print(f"\n  OLS (log TTR ~ country + reason + year_centred)")
        print(f"  R² = {res.rsquared:.3f}  |  Adj R² = {res.rsquared_adj:.3f}")
        coef_df = (
            res.summary2().tables[1]
            [lambda d: d.index.str.startswith("C(country_cat)")]
            .copy()
        )
        coef_df.index = coef_df.index.str.replace(r"C\(country_cat\)\[T\.", "", regex=True).str.rstrip("]")
        coef_df.columns = [c.strip() for c in coef_df.columns]
        coef_df = coef_df.rename(columns={"Coef.":"coef","[0.025":"ci_lo","0.975]":"ci_hi","P>|t|":"p"})
        coef_df = coef_df.sort_values("coef")
        coef_df["sig"] = coef_df["p"] < 0.05
 
        fig, ax = plt.subplots(figsize=(9, max(6, len(coef_df)*0.4)))
        colours = [PALETTE[1] if s else PALETTE[6] for s in coef_df["sig"]]
        ax.barh(coef_df.index, coef_df["coef"], color=colours, alpha=0.85)
        ax.errorbar(coef_df["coef"], coef_df.index,
                    xerr=[coef_df["coef"]-coef_df["ci_lo"],
                          coef_df["ci_hi"]-coef_df["coef"]],
                    fmt="none", color="#333", lw=1, capsize=3)
        ax.axvline(0, color="#555", lw=1.2, ls="--")
        ax.set_xlabel(f"OLS coefficient (log TTR, relative to {ref})")
        ax.set_title(
            "Country effect on time-to-retraction\n"
            "controlling for reason category and year\n"
            f"(R²={res.rsquared:.3f}, HC3 SEs; teal = p<0.05)"
        )
        sig_p = mpatches.Patch(color=PALETTE[1], alpha=0.85, label="p < 0.05")
        ns_p  = mpatches.Patch(color=PALETTE[6], alpha=0.85, label="p ≥ 0.05")
        ax.legend(handles=[sig_p, ns_p], fontsize=9)
        ax.grid(axis="x", visible=False); ax.grid(axis="y", visible=False)
        savefig("fig20c_country_ols_coefficients")
 
        md_base = smf.ols("log_ttr ~ reason_code + year_c", data=ols_df)
        res_base = md_base.fit()
        print(f"\n  R² without country dummies: {res_base.rsquared:.3f}")
        print(f"  R² with    country dummies: {res.rsquared:.3f}")
        print(f"  Δ R² attributable to country: {res.rsquared - res_base.rsquared:.3f}")
        print(f"  → Interpretation: most TTR variance is explained by reason, not country.")
 
    except Exception as e:
        print(f"  OLS skipped: {e}")
 
    # ── 4. Fig 20: Median TTR per country ─────────────────────────────────────
    med_ttr = (
        geo_ttr.groupby("country_name")["time_to_retraction_years"]
        .agg(["median","count"]).sort_values("median")
    )
    plot_data = pd.concat([med_ttr.head(15), med_ttr.tail(15)]).drop_duplicates()
    overall_med = geo_ttr["time_to_retraction_years"].median()
 
    fig, ax = plt.subplots(figsize=(9, 9))
    colours = [PALETTE[1] if v<=overall_med else PALETTE[4] for v in plot_data["median"]]
    ax.barh(plot_data.index, plot_data["median"], color=colours, alpha=0.85)
    for i, (idx, row) in enumerate(plot_data.iterrows()):
        ax.text(row["median"]+0.05, i, f"n={int(row['count']):,}",
                va="center", fontsize=8, color="#444")
    ax.axvline(overall_med, color="#555", lw=1.2, ls="--",
               label=f"Overall median: {overall_med:.1f} yr")
    ax.set_xlabel("Median time to retraction (years)")
    ax.set_title("Temporal lag by country\n(fastest ← 15 | slowest → 15; ≥50 papers)")
    fast = mpatches.Patch(color=PALETTE[1], alpha=0.85, label="≤ overall median")
    slow = mpatches.Patch(color=PALETTE[4], alpha=0.85, label="> overall median")
    ax.legend(handles=[fast, slow], fontsize=9)
    ax.grid(axis="x", visible=False); ax.grid(axis="y", visible=False)
    savefig("fig20_temporal_lag_country")
 
    top6_reasons = geo_ttr["primary_reason"].value_counts().index[:6].tolist()
    top6_countries = geo_ttr["iso3"].value_counts().index[:6].tolist()
    strat = (
        geo_ttr[geo_ttr["iso3"].isin(top6_countries) & geo_ttr["primary_reason"].isin(top6_reasons)]
        .groupby(["country_name","primary_reason"])["time_to_retraction_years"].median()
        .unstack(fill_value=np.nan)
    )
    fig, ax = plt.subplots(figsize=(12, 6))
    sns.heatmap(strat, cmap="RdYlGn_r", annot=True, fmt=".1f",
                linewidths=0.4, ax=ax,
                cbar_kws={"label":"Median TTR (years)","shrink":0.5})
    ax.set_title(
        "Median TTR by country × reason\n"
    )
    plt.xticks(rotation=35, ha="right", fontsize=9)
    savefig("fig20d_country_reason_ttr_heatmap")

SUBJECT ANALYSIS

In [ ]:
def phase5c_per_subject(df):
    print("\n══ Phase 5c — Per-subject analysis ══")
    set_style()
    if not BIOMEDICAL_SUBJECTS:
        print("  BIOMEDICAL_SUBJECTS empty — skipping."); return
 
    subj_long = (
        df.assign(SubjectList=df["Subject"].str.split(";")).explode("SubjectList")
    )
    subj_long["SubjectList"] = subj_long["SubjectList"].str.strip()
    pattern = "|".join(re.escape(s) for s in BIOMEDICAL_SUBJECTS)
    subj_long = subj_long[
        subj_long["SubjectList"].str.contains(pattern, case=False, na=False)
    ].copy()
 
    subject_list = subj_long["SubjectList"].value_counts().index.tolist()[:12]
    total = len(df)
 
    summary = []
    for s in subject_list:
        sub = subj_long[subj_long["SubjectList"]==s]
        summary.append({
            "subject": textwrap.shorten(s, 35),
            "count":   sub["Record ID"].nunique(),
            "median_ttr": sub["time_to_retraction_years"].dropna().median(),
        })
    sumdf = pd.DataFrame(summary).sort_values("count", ascending=True)
 
    fig, axes = plt.subplots(1, 2, figsize=(14, 7))
    axes[0].barh(sumdf["subject"], sumdf["count"], color=PALETTE[0], alpha=0.85)
    axes[0].set_xlabel("Number of retractions"); axes[0].set_title("Retraction count per subject")
    axes[0].grid(axis="x"); axes[0].grid(axis="y", visible=False)
    axes[1].barh(sumdf["subject"], sumdf["median_ttr"], color=PALETTE[2], alpha=0.85)
    axes[1].set_xlabel("Median TTR (years)"); axes[1].set_title("Median TTR per subject")
    axes[1].grid(axis="x"); axes[1].grid(axis="y", visible=False)
    savefig("fig21_subject_overview")
 
    sr = (
        subj_long[subj_long["SubjectList"].isin(subject_list)]
        .explode("ReasonCategories").rename(columns={"ReasonCategories":"Category"})
    )
    sr_ct  = sr.groupby(["SubjectList","Category"]).size().unstack(fill_value=0)
    sr_pct = sr_ct.div(sr_ct.sum(axis=1), axis=0) * 100
    sr_pct.index = [textwrap.shorten(s,30) for s in sr_pct.index]
    fig, ax = plt.subplots(figsize=(15, 7))
    sns.heatmap(sr_pct, cmap="Blues", annot=True, fmt=".0f",
                linewidths=0.4, ax=ax, cbar_kws={"label":"% of subject's retractions","shrink":0.5})
    ax.set_title("Retraction reason heatmap by subject area")
    plt.xticks(rotation=38, ha="right", fontsize=9); plt.yticks(fontsize=9)
    savefig("fig22_subject_reason_heatmap")
 
    top5_s = subject_list[:5]
    fig, ax = plt.subplots(figsize=(10, 5))
    for i, s in enumerate(top5_s):
        sy = (
            subj_long[subj_long["SubjectList"]==s]
            .groupby("retraction_year").size().reset_index(name="count")
            .query("1980 <= retraction_year <= 2026")
        )
        ax.plot(sy["retraction_year"], sy["count"],
                label=textwrap.shorten(s,30), color=PALETTE[i], lw=2.2, marker="o", ms=3)
    ax.set_xlabel("Year"); ax.set_ylabel("Number of retractions")
    ax.set_title("Annual retractions per subject"); ax.legend(fontsize=8.5, ncol=2)
    savefig("fig23_subject_annual_trends")
 
    print(f"\n  {'Subject':<40} {'Count':>7} {'% Total':>8} {'Median TTR':>12}")
    print("  " + "─"*70)
    for r in sorted(summary, key=lambda x: -x["count"]):
        print(f"  {r['subject']:<40} {r['count']:>7,} {r['count']/total*100:>7.1f}%"
              f" {r['median_ttr']:>11.1f} yr")

PUBMED CITATION COUNTS

In [ ]:
def phase6_citations(df, n_sample=5000):
    """
    Citation enrichment strategy (maximises coverage for 5k papers):
    1. iCite batch via PMID (fastest, most reliable for biomedical)
    2. OpenAlex BATCH endpoint via DOI (100 DOIs per call, not sequential)
    3. OpenAlex title-search fallback for remaining unmatched papers
 
    New analyses:
    A. Citation velocity: citations-per-year at time of retraction
    B. High-citation vs low-citation TTR comparison (Mann-Whitney)
    C. Post-retraction citation contamination estimate
    D. Relative Citation Ratio (RCR) distribution by reason category
    """
    import requests
    print("\n══ Phase 6 — Citation enrichment (iCite + OpenAlex batch) ══")
 
    citation_map = {}   
 
    # ── STEP 1: iCite via PMID ────────────────────────────────────────────────
    pmids = df["OriginalPaperPubMedID"].dropna().astype(int).unique()
    print(f"  iCite: {len(pmids):,} PMIDs…")
    for i in range(0, len(pmids), 100):
        ids = ",".join(str(x) for x in pmids[i:i+100])
        try:
            r = requests.get(
                "https://icite.od.nih.gov/api/pubs",
                params={"pmids":ids,"fields":"pmid,citation_count,relative_citation_ratio,year"},
                timeout=25,
            )
            r.raise_for_status()
            for rec in r.json().get("data",[]):
                citation_map[f"pmid:{rec['pmid']}"] = {
                    "citation_count": rec.get("citation_count"),
                    "rcr":            rec.get("relative_citation_ratio"),
                    "pub_year_icite": rec.get("year"),
                    "source":         "iCite",
                    "record_id":      None,
                }
        except Exception as e:
            print(f"  iCite chunk {i} failed: {e}")
    print(f"  iCite matched: {len(citation_map):,}")
 
    pmid_to_key = {int(k.split(":")[1]): k for k in citation_map}
    df["_cit_key"] = df["OriginalPaperPubMedID"].dropna().astype(int).map(pmid_to_key)
 
    # ── STEP 2: OpenAlex BATCH via DOI ────────────────────────────────────────
    missing_mask = df["_cit_key"].isna() & df["OriginalPaperDOI"].notna()
    dois_to_fetch = df.loc[missing_mask, ["Record ID","OriginalPaperDOI"]].dropna()
    dois_to_fetch = dois_to_fetch[dois_to_fetch["OriginalPaperDOI"].str.len() > 3]
 
    print(f"  OpenAlex batch DOI: {len(dois_to_fetch):,} unmatched…")
    CHUNK = 50   
    doi_cit_map = {}  
    for i in range(0, len(dois_to_fetch), CHUNK):
        chunk_dois = dois_to_fetch["OriginalPaperDOI"].iloc[i:i+CHUNK].tolist()
        filter_str = "|".join(f"doi:{quote(d)}" for d in chunk_dois)
        try:
            r = requests.get(
                "https://api.openalex.org/works",
                params={
                    "filter": f"doi:{('|doi:'.join(quote(d) for d in chunk_dois))}",
                    "per_page": CHUNK,
                    "select": "doi,cited_by_count",
                },
                headers={"User-Agent": "mailto:research@university.edu"},
                timeout=30,
            )
            if r.status_code == 200:
                for work in r.json().get("results", []):
                    raw_doi = (work.get("doi") or "").replace("https://doi.org/","").lower()
                    doi_cit_map[raw_doi] = work.get("cited_by_count")
        except Exception as e:
            if i == 0: print(f"  OpenAlex batch failed ({e}) — trying individual fallback")
 
    df["_doi_norm"] = df["OriginalPaperDOI"].str.lower().str.strip()
    df.loc[missing_mask, "_oa_cit"] = df.loc[missing_mask, "_doi_norm"].map(doi_cit_map)
    oa_matched = df["_oa_cit"].notna().sum()
    print(f"  OpenAlex batch matched: {oa_matched:,}")
 
    # ── STEP 3: OpenAlex title fallback (slow — use sparingly) ───────────────
    still_missing = df["_cit_key"].isna() & df["_oa_cit"].isna() & df["Title"].notna()
    titles_to_try = df.loc[still_missing].head(500)   # cap at 500 slow calls
    print(f"  OpenAlex title fallback: {len(titles_to_try):,} papers…")
    title_cit_map = {}
    for _, row in titles_to_try.iterrows():
        try:
            r = requests.get(
                "https://api.openalex.org/works",
                params={"search": row["Title"][:150], "per_page":1, "select":"title,cited_by_count"},
                headers={"User-Agent": "mailto:research@university.edu"},
                timeout=20,
            )
            if r.status_code == 200:
                res = r.json().get("results",[])
                if res:
                    title_cit_map[row["Record ID"]] = res[0].get("cited_by_count")
        except Exception:
            pass
    title_idx = df["Record ID"].isin(title_cit_map)
    df.loc[title_idx, "_oa_title_cit"] = df.loc[title_idx, "Record ID"].map(title_cit_map)
    print(f"  Title fallback matched: {title_idx.sum():,}")
 
    # ── Consolidate citation_count column ─────────────────────────────────────
    def _get_cit(row):
        if row["_cit_key"] in citation_map:
            return citation_map[row["_cit_key"]]["citation_count"]
        if pd.notna(row.get("_oa_cit")):
            return row["_oa_cit"]
        if pd.notna(row.get("_oa_title_cit")):
            return row["_oa_title_cit"]
        return np.nan
    def _get_rcr(row):
        if row["_cit_key"] in citation_map:
            return citation_map[row["_cit_key"]].get("rcr")
        return np.nan
 
    df["citation_count"] = df.apply(_get_cit, axis=1)
    df["rcr"]            = df.apply(_get_rcr, axis=1)
    total_matched = df["citation_count"].notna().sum()
    print(f"  Total matched: {total_matched:,} / {len(df):,} ({total_matched/len(df)*100:.1f}%)")
 
    df.drop(columns=[c for c in ["_cit_key","_doi_norm","_oa_cit","_oa_title_cit"]
                     if c in df.columns], inplace=True)
 
    set_style()
 
    # ── ANALYSIS A: Citation count vs TTR ──────────────────────────
    plot_df = (
        df[["citation_count","time_to_retraction_years"]].dropna()
        .query("0 <= time_to_retraction_years <= 25")
    )
    q97 = plot_df["citation_count"].quantile(0.97)
    plot_df = plot_df[plot_df["citation_count"] <= q97]
 
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.scatter(plot_df["citation_count"], plot_df["time_to_retraction_years"],
               alpha=0.22, s=12, color=PALETTE[0], rasterized=True)
    try:
        from statsmodels.nonparametric.smoothers_lowess import lowess
        sm = lowess(plot_df["time_to_retraction_years"], plot_df["citation_count"], frac=0.3)
        ax.plot(sm[:,0], sm[:,1], color=PALETTE[4], lw=2.5, label="Trend")
        ax.legend(fontsize=9)
    except ImportError:
        pass
    r, p = spearmanr(plot_df["citation_count"], plot_df["time_to_retraction_years"])
    ax.set_xlabel("Citation count"); ax.set_ylabel("Time to retraction (years)")
    ax.set_title(f"Citation count vs. TTR  (ρ={r:.3f}, p={p:.2e}, n={len(plot_df):,})\n")
                 #moderate positive correlation — more-cited papers take longer to retract")
    savefig("fig24_citations_vs_ttr")
 
    # ── ANALYSIS B: Citation VELOCITY (citations per year since publication) ──
    cit_vel = df.copy()
    cit_vel["years_since_pub"] = (
        pd.Timestamp("today") - cit_vel["OriginalPaperDate"]
    ).dt.days / 365.25
    cit_vel = cit_vel[cit_vel["years_since_pub"] > 0.5]
    cit_vel["citation_velocity"] = cit_vel["citation_count"] / cit_vel["years_since_pub"]
 
    vel_df = (
        cit_vel[["citation_velocity","time_to_retraction_years"]].dropna()
        .query("0 <= time_to_retraction_years <= 25")
    )
    q97v = vel_df["citation_velocity"].quantile(0.97)
    vel_df = vel_df[vel_df["citation_velocity"] <= q97v]
 
    rv, pv = spearmanr(vel_df["citation_velocity"], vel_df["time_to_retraction_years"])
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.scatter(vel_df["citation_velocity"], vel_df["time_to_retraction_years"],
               alpha=0.22, s=12, color=PALETTE[1], rasterized=True)
    try:
        sm = lowess(vel_df["time_to_retraction_years"], vel_df["citation_velocity"], frac=0.3)
        ax.plot(sm[:,0], sm[:,1], color=PALETTE[4], lw=2.5, label="Trend")
        ax.legend(fontsize=9)
    except Exception:
        pass
    ax.set_xlabel("Citation velocity (citations per year since publication)")
    ax.set_ylabel("Time to retraction (years)")
    ax.set_title(f"Citation velocity vs. TTR  (ρ={rv:.3f}, p={pv:.2e})\n")
    savefig("fig24b_citation_velocity_vs_ttr")
    print(f"  Citation velocity ρ={rv:.3f}, p={pv:.3e}")
 
    # ── ANALYSIS C: High-citation vs low-citation TTR (Mann-Whitney) ──────────
    med_cit = df["citation_count"].median()
    hi = df.loc[df["citation_count"] >  med_cit, "time_to_retraction_years"].dropna()
    lo = df.loc[df["citation_count"] <= med_cit, "time_to_retraction_years"].dropna()
    lo = lo[(lo>=0)&(lo<=30)]; hi = hi[(hi>=0)&(hi<=30)]
    U, p_mw = mannwhitneyu(hi, lo, alternative="two-sided")
    print(f"\n  High-citation (n={len(hi):,}) median TTR: {hi.median():.2f} yr")
    print(f"  Low-citation  (n={len(lo):,}) median TTR: {lo.median():.2f} yr")
    print(f"  Mann-Whitney U={U:.0f}, p={p_mw:.4e}")
 
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.violinplot([lo.values, hi.values], positions=[1,2], showmedians=True,
                  widths=0.6)
    ax.set_xticks([1, 2])
    ax.set_xticklabels([f"Low-citation\n(≤{med_cit:.0f}, n={len(lo):,})",
                        f"High-citation\n(>{med_cit:.0f}, n={len(hi):,})"])
    ax.set_ylabel("Time to retraction (years)")
    ax.set_title(
        f"TTR: high-citation vs. low-citation papers\n")
        
    
    ax.grid(axis="y", visible=False)
    savefig("fig24c_high_vs_low_citation_ttr")
 
    # ── ANALYSIS D: RCR distribution by reason (relative citation ratio) ──────
    if df["rcr"].notna().sum() >= 50:
        rcr_df = (
            df.explode("ReasonCategories")
            .rename(columns={"ReasonCategories":"Category"})
            [["Category","rcr"]].dropna()
            .query("rcr > 0 and rcr < 20")
        )
        top5_cats = rcr_df["Category"].value_counts().index[:5]
        rcr_df = rcr_df[rcr_df["Category"].isin(top5_cats)]
        rcr_df["Category_short"] = rcr_df["Category"].apply(lambda x: textwrap.shorten(x, 25))
 
        fig, ax = plt.subplots(figsize=(10, 5))
        sns.boxplot(data=rcr_df, x="Category_short", y="rcr",
                    palette=PALETTE[:5], linewidth=0.8, fliersize=2, ax=ax)
        ax.axhline(1.0, color="#888", lw=1.2, ls="--", label="RCR = 1 (field average)")
        ax.set_xlabel(""); ax.set_ylabel("Relative Citation Ratio (RCR)")
        ax.set_xticklabels(ax.get_xticklabels(), rotation=25, ha="right")
        ax.set_title("Field-normalized citation impact by retraction reason\n"
                     "(RCR > 1 = above field average, only from iCite)")
        ax.legend(fontsize=9)
        #savefig("fig24d_rcr_by_reason")
 
    return df

In [ ]:
def sleeping_beauty_analysis(df):
    """
    Identifies papers that accumulated citations for a long time before being
    retracted — the most dangerous retractions because they've had maximum
    influence time. Uses OpenAlex yearly citation data for papers with DOIs.
    """
    print("\n══ Sleeping beauty analysis ══")
    set_style()
 
    # ── Proxy metric: high citations × long TTR ───────────────────────────────
    sb = df[["Record ID","Title","citation_count","time_to_retraction_years",
             "ReasonCategories","Journal","retraction_year"]].dropna(
                subset=["citation_count","time_to_retraction_years"]
             ).copy()
    sb = sb.query("0 < time_to_retraction_years <= 30 and citation_count > 0")
 
    # Normalise each dimension to [0,1] and compute a "sleeping beauty score"
    sb["ttr_norm"] = (sb["time_to_retraction_years"] - sb["time_to_retraction_years"].min()) / \
                     (sb["time_to_retraction_years"].max() - sb["time_to_retraction_years"].min())
    q95 = sb["citation_count"].quantile(0.95)
    sb["cit_norm"] = (sb["citation_count"].clip(upper=q95) / q95)
    sb["sb_score"]  = (sb["ttr_norm"] * sb["cit_norm"]) ** 0.5   # geometric mean
 
    top_sb = sb.nlargest(20, "sb_score").copy()
    top_sb["reason"] = top_sb["ReasonCategories"].apply(lambda x: x[0] if x else "Unknown")
    top_sb["label"]  = top_sb["Title"].apply(lambda x: textwrap.shorten(str(x), 45))
 
    # ── Fig: Top sleeping beauties scatter ────────────────────────────────────
    fig, ax = plt.subplots(figsize=(9, 7))
    sc = ax.scatter(
        sb["time_to_retraction_years"], sb["citation_count"].clip(upper=q95),
        c=sb["sb_score"], cmap="YlOrRd", alpha=0.45, s=18, rasterized=True,
    )
    plt.colorbar(sc, ax=ax, label="Score (↑ = high TTR × high citations)")
 
    '''
    for _, row in top_sb.head(10).iterrows():
        ax.annotate(
            textwrap.shorten(str(row["Title"]), 30),
            (row["time_to_retraction_years"], min(row["citation_count"], q95)),
            fontsize=6.5, color="#333",
            arrowprops=dict(arrowstyle="-", color="#AAAAAA", lw=0.7),
            xytext=(row["time_to_retraction_years"]+0.5, min(row["citation_count"]*1.15, q95)),
        )
    '''
    ax.set_xlabel("Time to retraction (years)")
    ax.set_ylabel(f"Citation count (capped at 95th pct = {q95:.0f})")
    ax.set_title(
        "Papers that accumulated citations for years before retraction\n"
    )
    savefig("fig_sleeping_beauty_scatter")
 
    print("\n  Top 10 sleeping beauties:")
    print(f"  {'Score':>6}  {'TTR':>5}  {'Cit':>6}  Title")
    for _, row in top_sb.head(10).iterrows():
        print(f"  {row['sb_score']:.3f}  {row['time_to_retraction_years']:>5.1f}  "
              f"{int(row['citation_count']):>6,}  {row['label']}")
 
    # ── OpenAlex yearly citations for top 5 (if DOIs available) ──────────────
    try:
        import requests
        top5_dois = (
            df[df["Record ID"].isin(top_sb["Record ID"].head(5))]
            ["OriginalPaperDOI"].dropna().tolist()
        )
        print(f"\n  Fetching yearly citation counts for top 5 papers via OpenAlex…")
        for doi in top5_dois:
            try:
                r = requests.get(
                    f"https://api.openalex.org/works/https://doi.org/{quote(str(doi))}",
                    params={"select":"title,counts_by_year,cited_by_count"},
                    headers={"User-Agent":"mailto:research@university.edu"},
                    timeout=25,
                )
                if r.status_code != 200: continue
                j = r.json()
                cby = j.get("counts_by_year",[])
                if not cby: continue
                yr_df = pd.DataFrame(cby).sort_values("year")
                fig, ax = plt.subplots(figsize=(8, 4))
                ax.bar(yr_df["year"], yr_df["cited_by_count"],
                       color=PALETTE[0], alpha=0.8)
                retraction_year = df.loc[df["OriginalPaperDOI"]==doi,"retraction_year"].iloc[0] \
                                  if (df["OriginalPaperDOI"]==doi).any() else None
                if retraction_year:
                    ax.axvline(retraction_year, color=PALETTE[4], lw=2, ls="--",
                               label=f"Retracted: {int(retraction_year)}")
                    ax.legend(fontsize=9)
                title_short = textwrap.shorten(str(j.get("title",doi)), 60)
                ax.set_xlabel("Year"); ax.set_ylabel("Citations in year")
                ax.set_title(f"Citation trajectory \n{title_short}")
                safe_doi = re.sub(r"[/\\\s]","_", str(doi))[:40]
                savefig(f"fig_sleeping_beauty_{safe_doi}")
            except Exception as e:
                print(f"  Failed for {doi}: {e}")
    except ImportError:
        pass
 
    return top_sb

In [ ]:
def post_retraction_citation_analysis(df):
    """
    For papers with OpenAlex yearly citation data:
    - Compute pre-retraction citation rate (citations/year before retraction)
    - Compute post-retraction citation rate (citations/year after retraction)
    - The ratio tells you how much contamination persists.
 
    For papers without time-series: use total citation count as a proxy
    for "contamination depth" and analyse by reason/journal.
    """
    print("\n══ Post-retraction citation contamination ══")
    import requests
    set_style()
 
    # ── Static proxy: total citation count at retraction ─────────────────────
    cit_df = df[["citation_count","time_to_retraction_years","ReasonCategories",
                 "retraction_year","Journal","is_hindawi"]].dropna(
                    subset=["citation_count","time_to_retraction_years"]
                 ).copy()
    cit_df = cit_df.query("0 < time_to_retraction_years <= 30 and citation_count > 0")
    cit_df["primary_reason"] = cit_df["ReasonCategories"].apply(lambda x: x[0])
 
    top6 = cit_df["primary_reason"].value_counts().index[:6]
    cit_cat = cit_df[cit_df["primary_reason"].isin(top6)]
 
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.boxplot(data=cit_cat,
                x="primary_reason", y="citation_count",
                order=top6, palette=PALETTE[:6],
                linewidth=0.8, fliersize=2, ax=ax)
    ax.set_yscale("log")
    ax.set_xticklabels([textwrap.shorten(t.get_text(), 22)
                        for t in ax.get_xticklabels()], rotation=28, ha="right")
    ax.set_xlabel(""); ax.set_ylabel("Citation count at retraction (log scale)")
    ax.set_title("Citation count at retraction per reason\n")
                 #higher = more papers built on this work before retraction)"
    savefig("fig_post_retraction_contamination_by_reason")
 
    sample = (
        df[df["OriginalPaperDOI"].notna() & df["retraction_year"].notna()]
        .nlargest(50, "citation_count")
    )
    print(f"  Fetching yearly citation curves for top-50 most-cited retracted papers…")
 
    pre_rates, post_rates, labels = [], [], []
    for _, row in sample.iterrows():
        doi = str(row["OriginalPaperDOI"])
        ret_year = int(row["retraction_year"]) if pd.notna(row["retraction_year"]) else None
        if not ret_year: continue
        try:
            r = requests.get(
                f"https://api.openalex.org/works/https://doi.org/{quote(doi)}",
                params={"select":"counts_by_year"},
                headers={"User-Agent":"mailto:research@university.edu"},
                timeout=20,
            )
            if r.status_code != 200: continue
            cby = r.json().get("counts_by_year", [])
            if not cby: continue
            yr_df = pd.DataFrame(cby)
            pre  = yr_df[yr_df["year"] <  ret_year]["cited_by_count"].sum()
            post = yr_df[yr_df["year"] >= ret_year]["cited_by_count"].sum()
            n_pre  = max(1, ret_year - int(row["pub_year"]) if pd.notna(row["pub_year"]) else 1)
            n_post = max(1, 2025 - ret_year)
            pre_rates.append(pre  / n_pre)
            post_rates.append(post / n_post)
            labels.append(textwrap.shorten(str(row["Title"]), 35))
        except Exception:
            pass
 
    if pre_rates:
        rate_df = pd.DataFrame({"pre": pre_rates, "post": post_rates, "label": labels})
        
  
        sample_reset = sample.reset_index(drop=True)
        rate_df["doi"]            = sample_reset["OriginalPaperDOI"].iloc[:len(rate_df)].values
        rate_df["full_title"]     = sample_reset["Title"].iloc[:len(rate_df)].values
        rate_df["journal"]        = sample_reset["Journal"].iloc[:len(rate_df)].values
        rate_df["retraction_year"]= sample_reset["retraction_year"].iloc[:len(rate_df)].values
        rate_df["pub_year"]       = sample_reset["pub_year"].iloc[:len(rate_df)].values
        rate_df["primary_reason"] = sample_reset["ReasonCategories"].iloc[:len(rate_df)].apply(lambda x: x[0])
        rate_df["citation_count"] = sample_reset["citation_count"].iloc[:len(rate_df)].values
        rate_df["ratio"]          = rate_df["post"] / (rate_df["pre"] + 0.01)

        # ── Papers ABOVE the diagonal (post rate > pre rate) ──────────────────────
        above_diag = (
            rate_df[rate_df["post"] > rate_df["pre"]]
            .sort_values("ratio", ascending=False)
            .copy()
        )

        print(f"\n  Papers still cited FASTER after retraction: {len(above_diag)} / {len(rate_df)}")
        print(f"  {'Ratio':>6}  {'Pre':>8}  {'Post':>8}  {'Ret.Yr':>6}  Title")
        print("  " + "─" * 90)
        for _, row in above_diag.iterrows():
            print(f"  {row['ratio']:>6.2f}  {row['pre']:>8.1f}  {row['post']:>8.1f}  "
                f"{int(row['retraction_year']):>6}  {str(row['full_title'])[:60]}")

        above_diag[[
            "full_title", "journal", "primary_reason",
            "pub_year", "retraction_year",
            "citation_count", "pre", "post", "ratio", "doi"
        ]].to_csv(f"{OUTPUT_DIR}/papers_above_diagonal.csv", index=False)
        print(f"\n  → papers_above_diagonal.csv saved to {OUTPUT_DIR}/")

        fig, ax = plt.subplots(figsize=(9, 7))
        
        colours = ["#E76F51" if p > r else "#2A9D8F" 
                for p, r in zip(rate_df["post"], rate_df["pre"])]
        sc = ax.scatter(rate_df["pre"], rate_df["post"],
                        c=colours, alpha=0.80, s=60, zorder=3)
        
        max_val = rate_df[["pre", "post"]].max().max() * 1.05
        ax.plot([0, max_val], [0, max_val],
                color="#888", lw=1.2, ls="--", label="Pre = post rate", zorder=2)
        
        for _, row in above_diag.head(8).iterrows():
            ax.annotate(
                textwrap.shorten(str(row["full_title"]), 28),
                (row["pre"], row["post"]),
                fontsize=6.5, color="#333",
                xytext=(row["pre"] + max_val * 0.02, row["post"] + max_val * 0.01),
                arrowprops=dict(arrowstyle="-", color="#BBBBBB", lw=0.8),
            )
        
        # Legend patches
        above_patch = mpatches.Patch(color="#E76F51", alpha=0.8,
                                    label=f"Above diagonal (n={len(above_diag)}) — still cited faster")
        below_patch = mpatches.Patch(color="#2A9D8F", alpha=0.8,
                                    label=f"Below diagonal (n={len(rate_df)-len(above_diag)}) — citation rate dropped")
        ax.legend(handles=[above_patch, below_patch], fontsize=9, loc="upper left")

        ax.set_xlabel("Pre-retraction citation rate (cit/yr)")
        ax.set_ylabel("Post-retraction citation rate (cit/yr)")
        ax.set_title(
            "Post vs. pre-retraction citation rate — top 50 most-cited papers\n"
            f"Red = still cited faster after retraction ({len(above_diag)}/{len(rate_df)} papers)"
        )
        savefig("fig_post_retraction_pre_vs_post_rate")
 
    return df

In [ ]:
def gender_analysis(df):
    """
    Infers gender of first and last authors using Genderize.io API (free tier:
    1,000 names/day) with a local gender-guesser fallback.
    Analyses:
    - Retraction count by inferred gender of first / last author
    - TTR by gender
    - Reason profile by gender
    Frame carefully: this is about structural patterns, not individual blame.
    """
    print("\n══ Gender analysis ══")
    set_style()
 
    def _parse_first_name(author_string, position="first"):
        """Extract first name of first or last author from semicolon-separated list."""
        if pd.isna(author_string): return None
        parts = [a.strip() for a in str(author_string).split(";") if a.strip()]
        if not parts: return None
        target = parts[0] if position=="first" else parts[-1]
        # Handle "LastName, FirstName" or "FirstName LastName"
        if "," in target:
            name_parts = target.split(",")
            first = name_parts[1].strip().split()[0] if len(name_parts)>1 else None
        else:
            name_parts = target.strip().split()
            first = name_parts[0] if name_parts else None
        return first if first and len(first) > 1 else None
 
    df["first_author_firstname"] = df["Author"].apply(lambda x: _parse_first_name(x,"first"))
    df["last_author_firstname"]  = df["Author"].apply(lambda x: _parse_first_name(x,"last"))
 
    gender_cache = {}
 
    def _gender_guesser(name):
        try:
            import gender_guesser.detector as gd
            d = gd.Detector(case_sensitive=False)
            result = d.get_gender(name)
            if result in ("male","mostly_male"):   return "M"
            if result in ("female","mostly_female"): return "F"
            return "U"
        except ImportError:
            return None
 
    def _genderize_batch(names):
        """Call Genderize.io for up to 10 names at once."""
        try:
            import requests
            params = "&".join(f"name[]={n}" for n in names[:10])
            r = requests.get(f"https://api.genderize.io?{params}", timeout=15)
            if r.status_code != 200: return {}
            return {
                item["name"].lower(): ("M" if item["gender"]=="male" else
                                        "F" if item["gender"]=="female" else "U")
                for item in r.json() if item.get("gender")
            }
        except Exception:
            return {}
 
    def lookup_gender(name):
        if pd.isna(name): return "U"
        name = str(name).strip()
        if name in gender_cache: return gender_cache[name]
        g = _gender_guesser(name)
        if g and g != "U":
            gender_cache[name] = g; return g
        return "U"   # batch genderize done below
 
    all_names = (
        pd.concat([df["first_author_firstname"], df["last_author_firstname"]])
        .dropna().unique().tolist()
    )
    print(f"  Looking up gender for {len(all_names):,} unique first names…")
    for n in all_names:
        lookup_gender(n)
    unknowns = [n for n in all_names if gender_cache.get(n,"U") == "U"]
    print(f"  Unresolved after offline: {len(unknowns):,} — calling Genderize.io…")
    for i in range(0, min(len(unknowns), 500), 10):
        batch = unknowns[i:i+10]
        result = _genderize_batch(batch)
        for n, g in result.items():
            gender_cache[n.title()] = g
            gender_cache[n] = g
 
    df["first_author_gender"] = df["first_author_firstname"].apply(lookup_gender)
    df["last_author_gender"]  = df["last_author_firstname"].apply(lookup_gender)
 
    resolved = (df["first_author_gender"].isin(["M","F"])).mean()
    print(f"  Gender resolved for {resolved*100:.1f}% of first authors")
    print(df["first_author_gender"].value_counts().to_string())
 
    g_df = df[df["first_author_gender"].isin(["M","F"])].copy()
    if len(g_df) < 100:
        print("  Too few resolved genders — skipping gender figures."); return df
 
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    gender_count = g_df["first_author_gender"].value_counts()
    axes[0].bar(["Male first author","Female first author"],
                [gender_count.get("M",0), gender_count.get("F",0)],
                color=[PALETTE[0], PALETTE[4]], alpha=0.85)
    axes[0].set_ylabel("Number of retractions")
    axes[0].set_title("Retractions by first-author gender")
    axes[0].grid(axis="y", visible=False); axes[0].grid(axis="x", visible=False)
    
 
    ttr_g = g_df[["first_author_gender","time_to_retraction_years"]].dropna()
    ttr_g = ttr_g.query("0 <= time_to_retraction_years <= 30")
    sns.violinplot(data=ttr_g, x="first_author_gender", y="time_to_retraction_years",
                   palette=[PALETTE[0], PALETTE[4]], inner="quartile",
                   linewidth=0.8, ax=axes[1])
    axes[1].set_xticklabels(["Male first author","Female first author"])
    axes[1].set_ylabel("Time to retraction (years)")
    axes[1].set_title("Time to retraction by first-author gender")
    m_ttr = ttr_g[ttr_g["first_author_gender"]=="M"]["time_to_retraction_years"]
    f_ttr = ttr_g[ttr_g["first_author_gender"]=="F"]["time_to_retraction_years"]
    from scipy.stats import mannwhitneyu as mwu
    U, p = mwu(m_ttr, f_ttr, alternative="two-sided")
    savefig("fig_gender_retraction_ttr")
 
    gr = (
        g_df.explode("ReasonCategories").rename(columns={"ReasonCategories":"Category"})
        [["first_author_gender","Category"]]
    )
    top8 = gr["Category"].value_counts().index[:8]
    gr = gr[gr["Category"].isin(top8)]
    ct = pd.crosstab(gr["first_author_gender"], gr["Category"], normalize="index") * 100
 
    fig, ax = plt.subplots(figsize=(12, 5))
    ct.T.plot(kind="bar", ax=ax, color=[PALETTE[0],PALETTE[4]], alpha=0.85, width=0.7)
    ax.set_xticklabels([textwrap.shorten(t.get_text(),20) for t in ax.get_xticklabels()],
                       rotation=35, ha="right")
    ax.set_ylabel("% of gender's retractions")
    ax.set_title("Retraction reason profile by first-author gender\n"
                 "(% within each gender")
    ax.legend(["Male first author","Female first author"], fontsize=9)
    ax.grid(axis="y", visible=False); ax.grid(axis="x", visible=False)
    savefig("fig_gender_reason_profile")
 
    return df

MAIN

In [ ]:
if __name__ == "__main__":
    Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
    set_style()
 
    df = phase1_load(DATA_PATH)
    if df is None:
        raise SystemExit("Populate BIOMEDICAL_SUBJECTS and set EXPLORE_SUBJECTS=False.")
  
    df = phase2_clean(df)
    reasons_long = build_reasons_long(df)   # built once, passed everywhere
    df[["Title","OriginalPaperDOI"]].dropna().to_csv("retracted_titles.csv", index=False)

    phase3a_temporal(df)
    phase3b_reasons(df, reasons_long)
    geo_result = phase3c_geographic(df)
    if geo_result[0] is not None:
        country_counts, geo = geo_result

    #phase3c_geographicv2(df)
    
    #print(geo.loc[geo["iso3"].eq("KNA"), ["Record ID", "Country", "iso3"]].head(20).to_string(index=False))

    phase4_journals_authors(df)
    phase5_statistics(df, reasons_long)     
    phase5b_temporal_lag_by_country(df)
    phase5c_per_subject(df)
    df = phase6_citations(df, n_sample=5000)
    top_sb = sleeping_beauty_analysis(df)
    df = post_retraction_citation_analysis(df)
    #df = gender_analysis(df)
 
    print(f"\nDone. All figures in → {OUTPUT_DIR}/\n")


══════════════════════════════════════════
  Phase 1 — Data ingestion & subject filter
══════════════════════════════════════════
  Total records in CSV          :   69,626
  After RetractionNature filter :   64,444
  After biomedical subject filter:   4,970
  After biomedical subject filter:   4,970

══════════════════════════════════════════
  Phase 2 — Cleaning & feature engineering
══════════════════════════════════════════
  Dropped 0 rows with negative TTR
  Clean records: 4,970  |  Hindawi: 1,288

══ Phase 3a — Temporal trends ══
    → fig01_annual_retractions.pdf / .png
    → fig02_time_to_retraction.pdf / .png

══ Phase 3b — Reasons analysis ══
    → fig04_reason_frequency.pdf / .png
    → fig05_reason_cooccurrence.pdf / .png
    → fig06_reason_trends.pdf / .png

══ Phase 3c — Geographic analysis ══
    → fig07_choropleth.html  (sqrt color scale, real counts in hover)
    → fig08a_top20_raw_counts.pdf / .png
  Major research nations (≥100,000 docs): 61
    → fig08b_top20_norm